In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os
import tensorflow as tf


train = pd.read_csv("../emnist-balanced-train.csv", header=None)
test = pd.read_csv("../emnist-balanced-test.csv", header=None)



In [17]:
x_train = train.iloc[:, 1:].values
y_train = train.iloc[:, 0].values

x_test = test.iloc[:, 1:].values
y_test = test.iloc[:, 0].values

In [18]:
print(x_train.shape, y_train.shape)

(112800, 784) (112800,)


In [19]:
print(test.shape, train.shape)

(18800, 785) (112800, 785)


In [20]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [21]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [22]:
y_train_onehot = tf.keras.utils.to_categorical(y_train_encoded)
y_test_onehot = tf.keras.utils.to_categorical(y_test_encoded)

In [23]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [24]:
model = tf.keras.Sequential([

    tf.keras.layers.Dense(512, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(len(le.classes_), activation="softmax")
])

/home/busungen/Documents/Deep Learning Project/Deep-Learning-Grupp-1/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 47)             │         6,063 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 572,207 (2.18 MB)

 Trainable params: 572,207 (2.18 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [27]:
ANN_model = model.fit(
    x_train, y_train_onehot,
    validation_data=(x_test, y_test_onehot),
    epochs=40,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6065 - loss: 1.3417 - val_accuracy: 0.7856 - val_loss: 0.6825
Epoch 2/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7507 - loss: 0.7838 - val_accuracy: 0.8131 - val_loss: 0.5732
Epoch 3/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7816 - loss: 0.6704 - val_accuracy: 0.8281 - val_loss: 0.5187
Epoch 4/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7949 - loss: 0.6171 - val_accuracy: 0.8315 - val_loss: 0.5054
Epoch 5/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8066 - loss: 0.5795 - val_accuracy: 0.8388 - val_loss: 0.4824
Epoch 6/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8148 - loss: 0.5513 - val_accuracy: 0.8362 - val_loss: 0.4749
Epoch 7/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8200 - loss: 0.5301 - val_accuracy: 0.8458 - val_loss: 0.4478
Epoch 8/40
882/882 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8262 - loss: 0.5083 - val_accuracy: 0.

In [28]:
model.save("../trained_models/ann_model.keras")
joblib.dump(le, "../trained_models/label_encoded.pk1")

['../trained_models/label_encoded.pk1']